# Ejercicio 3 — Modelos de Clasificación con Dataset Propio
**Curso:** Ingeniería del Conocimiento (ISO56B)
**Periodo:** 2026-II | Universidad Nacional del Centro del Perú
**Dataset elegido:** [Brawl Stars — Rankings, Players and Clubs](https://www.kaggle.com/datasets/albertovidalrod/brawl-stars-rankings-players-and-clubs) (Kaggle)

---

## Objetivo
Replicar el flujo de la Sesión 02 (EDA → preprocesamiento → 4 modelos de clasificación → comparación) usando un dataset distinto al de Wine Quality, siguiendo la consigna del **Ejercicio 3**: dataset con ≥500 registros y ≥5 features.

## Sobre el dataset
El dataset original trae, entre otras cosas, un archivo `global_player_info.csv` por cada snapshot mensual con estadísticas detalladas de los **200 mejores jugadores del mundo** de Brawl Stars: nivel de experiencia, victorias, y el poder/rango/trofeos de cada uno de los ~70 personajes ("brawlers") que puede usar un jugador.

Como cada snapshot mensual solo trae 200 filas, **combinamos 6 snapshots mensuales** (marzo, agosto, septiembre, octubre, noviembre 2023 y el snapshot "actual") para llegar a **1200 registros**, y construimos features agregadas a partir de las ~70 columnas por-brawler (que de otra forma serían demasiado dispersas para modelar directamente).

> ⚠️ **Limitación a tener en cuenta:** el ranking global cambia poco de mes a mes, por lo que varios jugadores aparecen repetidos en más de un snapshot (con estadísticas ligeramente distintas). Tratamos cada fila (jugador-mes) como una observación independiente, lo cual es razonable para fines pedagógicos pero introduce cierta correlación entre observaciones que un estudio riguroso debería controlar (por ejemplo, con validación agrupada por `tag` de jugador).

---

## 1. Configuración del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay,
    accuracy_score, f1_score
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
print('Entorno configurado correctamente.')

---
## 2. Descarga y carga del dataset

Descargamos el dataset con `kagglehub` (igual que en el snippet compartido) y luego cargamos y combinamos los 6 snapshots de `global_player_info.csv`.

In [ ]:
import kagglehub

# Descargar la última versión del dataset
path = kagglehub.dataset_download("albertovidalrod/brawl-stars-rankings-players-and-clubs")
print("Path to dataset files:", path)

In [ ]:
import os

# Snapshots mensuales disponibles en el dataset (ajustar si kagglehub trae otros)
snapshot_paths = {
    'current': 'global_player_info.csv',
    'mar2023': 'month data - March 2023/global_player_info.csv',
    'aug2023': 'month data - August 2023/global_player_info.csv',
    'sep2023': 'month data - September 2023/global_player_info.csv',
    'oct2023': 'month data - October 2023/global_player_info.csv',
    'nov2023': 'month data - November 2023/global_player_info.csv',
}

dfs = []
for snapshot, rel_path in snapshot_paths.items():
    full_path = os.path.join(path, rel_path)
    if os.path.exists(full_path):
        d = pd.read_csv(full_path)
        d['snapshot'] = snapshot
        dfs.append(d)
    else:
        print(f'No encontrado: {full_path}')

df_raw = pd.concat(dfs, ignore_index=True)
print(f'Dataset combinado: {df_raw.shape[0]} registros, {df_raw.shape[1]} columnas')
print(f'Snapshots incluidos: {df_raw["snapshot"].value_counts().to_dict()}')
df_raw.head()

---
## 3. Ingeniería de features

Las columnas por-brawler (`<brawler>_power`, `<brawler>_rank`, `<brawler>_trophies`, `<brawler>_highestTrophies`) son demasiado numerosas (~290 columnas) y dispersas para usarlas directamente. Las resumimos en **features agregadas** que describen el perfil general del jugador, y las combinamos con sus estadísticas globales.

In [ ]:
power_cols   = [c for c in df_raw.columns if c.endswith('_power')]
rank_cols    = [c for c in df_raw.columns if c.endswith('_rank')]
trophy_cols  = [c for c in df_raw.columns if c.endswith('_trophies') and c != 'trophies']

print(f'Columnas de poder: {len(power_cols)} | rango: {len(rank_cols)} | trofeos por brawler: {len(trophy_cols)}')

df = pd.DataFrame(index=df_raw.index)

# Features agregadas del perfil "de brawlers"
df['avg_brawler_power']    = df_raw[power_cols].mean(axis=1)
df['avg_brawler_rank']     = df_raw[rank_cols].mean(axis=1)
df['avg_brawler_trophies'] = df_raw[trophy_cols].mean(axis=1)
df['n_brawlers_owned']     = df_raw[power_cols].notna().sum(axis=1)
df['n_maxed_brawlers']     = (df_raw[rank_cols] >= 30).sum(axis=1)
df['pct_maxed_brawlers']   = df['n_maxed_brawlers'] / df['n_brawlers_owned']

# Features de actividad / progreso general
df['expLevel']         = df_raw['expLevel']
df['expPoints']        = df_raw['expPoints']
df['3vs3Victories']    = df_raw['3vs3Victories']
df['soloVictories']    = df_raw['soloVictories']
df['duoVictories']     = df_raw['duoVictories']
df['total_victories']  = df['3vs3Victories'] + df['soloVictories'] + df['duoVictories']
df['solo_duo_ratio']   = df['soloVictories'] / (df['duoVictories'] + 1)

# Variable auxiliar (no se usa como feature, solo para construir el target)
df['trophies'] = df_raw['trophies']

print(f'\nDataset final: {df.shape[0]} registros, {df.shape[1]} columnas (incluye trophies auxiliar)')
print(df.isna().sum().sum(), 'valores nulos totales')

---
## 4. Construcción del target de clasificación

Al igual que en la Sesión 02 binarizamos una variable continua para crear un problema de **clasificación binaria**. Aquí usamos los **trofeos totales** del jugador:

- **Clase 1 (Élite):** trofeos en el cuartil superior (top 25%)
- **Clase 0 (Pro):** el resto (75% restante)

Esta binarización, igual que en el caso del vino, genera un **problema desbalanceado (~3:1)**, lo que añade un reto realista al modelado.

In [ ]:
thresh = df['trophies'].quantile(0.75)
df['elite_label'] = (df['trophies'] >= thresh).astype(int)

print(f'Umbral de trofeos (percentil 75): {thresh:,.0f}')
print(df['elite_label'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(df['trophies'], bins=30, ax=axes[0], color='#3498db')
axes[0].axvline(thresh, color='red', linestyle='--', label=f'Umbral = {thresh:,.0f}')
axes[0].set_title('Distribución de trofeos totales')
axes[0].legend()

counts = df['elite_label'].value_counts()
axes[1].bar(['Pro (0)', 'Élite (1)'], counts.values, color=['#3498db', '#e74c3c'])
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 10, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
axes[1].set_title('Clasificación binaria (umbral: percentil 75 de trofeos)')
axes[1].set_ylabel('Cantidad')

plt.tight_layout()
plt.show()

print(f'\nRatio de desbalance: {counts[0]/counts[1]:.1f}:1 (Pro vs Élite)')

---
## 5. Análisis exploratorio orientado a clasificación

In [ ]:
features = ['expLevel','expPoints','3vs3Victories','soloVictories','duoVictories',
            'total_victories','solo_duo_ratio','avg_brawler_power','avg_brawler_rank',
            'avg_brawler_trophies','n_brawlers_owned','pct_maxed_brawlers']

# 5.1  Estadísticas descriptivas por clase
comparison = df.groupby('elite_label')[features].mean().T
comparison.columns = ['Pro (0)', 'Élite (1)']
comparison['Diferencia %'] = ((comparison['Élite (1)'] - comparison['Pro (0)']) / comparison['Pro (0)'].abs() * 100).round(1)
comparison.style.background_gradient(subset=['Diferencia %'], cmap='RdYlGn')

In [ ]:
# 5.2  Distribución de features por clase (boxplots)
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.boxplot(data=df, x='elite_label', y=col, ax=axes[i],
                palette=['#3498db', '#e74c3c'], hue='elite_label', legend=False)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_xticklabels(['Pro', 'Élite'])

fig.suptitle('Distribución de features por clase', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 5.3  Correlación entre features
plt.figure(figsize=(9, 7))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Matriz de correlación — Features')
plt.tight_layout()
plt.show()

---
## 6. Preparación de datos

In [ ]:
# 6.1  Separación features / target
X = df[features].copy()
y = df['elite_label'].copy()

# 6.2  Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]} muestras  |  Test: {X_test.shape[0]} muestras')
print(f'Proporción clase Élite en train: {y_train.mean():.3f}')
print(f'Proporción clase Élite en test:  {y_test.mean():.3f}')

In [ ]:
# 6.3  Verificación de escalas (justifica la estandarización)
print('Rangos de las features (train):')
print('='*55)
for col in features:
    mn, mx = X_train[col].min(), X_train[col].max()
    print(f'{col:25s}  [{mn:12.2f}, {mx:12.2f}]  rango: {mx-mn:.2f}')

print('\nLas escalas difieren en varios ordenes de magnitud (ej. expPoints vs. pct_maxed_brawlers).')
print('Esto afecta directamente a SVM y KNN (basados en distancias).')

---
## 7. Modelo 1 — Regresión Logística

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                               class_weight='balanced'))
])

scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring='f1')
print(f'Logistic Regression - F1 CV: {scores_lr.mean():.4f} +/- {scores_lr.std():.4f}')

pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

In [ ]:
coefs = pd.Series(
    pipe_lr.named_steps['clf'].coef_[0],
    index=features
).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coefs.values]
coefs.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Regresion Logistica - Coeficientes estandarizados')
ax.set_xlabel('Coeficiente (impacto en la prediccion)')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

---
## 8. Modelo 2 — Árbol de Decisión

In [ ]:
pipe_dt = Pipeline([
    ('clf', DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=10,
        class_weight='balanced', random_state=RANDOM_STATE
    ))
])

scores_dt = cross_val_score(pipe_dt, X_train, y_train, cv=cv, scoring='f1')
print(f'Decision Tree - F1 CV: {scores_dt.mean():.4f} +/- {scores_dt.std():.4f}')

pipe_dt.fit(X_train, y_train)
y_pred_dt = pipe_dt.predict(X_test)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    pipe_dt.named_steps['clf'],
    feature_names=features,
    class_names=['Pro', 'Elite'],
    filled=True, rounded=True, fontsize=8, ax=ax,
    max_depth=3
)
ax.set_title('Arbol de Decision (primeros 3 niveles)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(
    pipe_dt.named_steps['clf'].feature_importances_,
    index=features
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='#3498db')
ax.set_title('Arbol de Decision - Importancia de features (Gini)')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

---
## 9. Modelo 3 — Support Vector Machine (SVM)

In [ ]:
kernels = ['linear', 'rbf', 'poly']
svm_results = {}

for kernel in kernels:
    pipe_svm_k = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel=kernel, class_weight='balanced',
                     random_state=RANDOM_STATE, probability=True))
    ])
    scores = cross_val_score(pipe_svm_k, X_train, y_train, cv=cv, scoring='f1')
    svm_results[kernel] = scores
    print(f'SVM ({kernel:6s}) - F1 CV: {scores.mean():.4f} +/- {scores.std():.4f}')

best_kernel = max(svm_results, key=lambda k: svm_results[k].mean())
print(f'\nMejor kernel: {best_kernel}')

In [ ]:
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel=best_kernel, class_weight='balanced',
                 random_state=RANDOM_STATE, probability=True))
])

pipe_svm.fit(X_train, y_train)
y_pred_svm = pipe_svm.predict(X_test)
scores_svm = svm_results[best_kernel]

In [ ]:
svm_no_scale = SVC(kernel=best_kernel, class_weight='balanced', random_state=RANDOM_STATE)
scores_no_scale = cross_val_score(svm_no_scale, X_train, y_train, cv=cv, scoring='f1')

print('Impacto de la estandarizacion en SVM:')
print(f'  CON StandardScaler: F1 = {scores_svm.mean():.4f}')
print(f'  SIN StandardScaler: F1 = {scores_no_scale.mean():.4f}')
print(f'  Diferencia: {(scores_svm.mean() - scores_no_scale.mean())*100:+.2f} puntos porcentuales')

---
## 10. Modelo 4 — K-Nearest Neighbors (KNN)

In [ ]:
k_range = range(1, 31)
k_scores = []

for k in k_range:
    pipe_knn_k = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(pipe_knn_k, X_train, y_train, cv=cv, scoring='f1')
    k_scores.append(scores.mean())

best_k = k_range[np.argmax(k_scores)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, k_scores, 'o-', markersize=4, color='#8e44ad')
ax.axvline(x=best_k, color='red', linestyle='--', alpha=0.7,
           label=f'Mejor K = {best_k} (F1 = {max(k_scores):.4f})')
ax.set_xlabel('Numero de vecinos (K)')
ax.set_ylabel('F1-Score (CV)')
ax.set_title('KNN - Busqueda del K optimo')
ax.legend()
plt.tight_layout()
plt.show()

print(f'K optimo: {best_k}')

In [ ]:
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', KNeighborsClassifier(n_neighbors=best_k))
])

scores_knn = cross_val_score(pipe_knn, X_train, y_train, cv=cv, scoring='f1')
print(f'KNN (K={best_k}) - F1 CV: {scores_knn.mean():.4f} +/- {scores_knn.std():.4f}')

pipe_knn.fit(X_train, y_train)
y_pred_knn = pipe_knn.predict(X_test)

---
## 11. Análisis comparativo de modelos

In [ ]:
models = {
    'Logistic Regression': (pipe_lr, y_pred_lr, scores_lr),
    'Decision Tree':       (pipe_dt, y_pred_dt, scores_dt),
    f'SVM ({best_kernel})': (pipe_svm, y_pred_svm, scores_svm),
    f'KNN (K={best_k})':   (pipe_knn, y_pred_knn, scores_knn)
}

summary = []
for name, (pipe, y_pred, cv_scores) in models.items():
    summary.append({
        'Modelo': name,
        'F1 CV (mean)': f'{cv_scores.mean():.4f}',
        'F1 CV (std)': f'{cv_scores.std():.4f}',
        'Accuracy Test': f'{accuracy_score(y_test, y_pred):.4f}',
        'F1 Test': f'{f1_score(y_test, y_pred):.4f}'
    })

summary_df = pd.DataFrame(summary)
print('='*80)
print('COMPARACION DE MODELOS')
print('='*80)
summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cv_data = [scores_lr, scores_dt, scores_svm, scores_knn]
labels = list(models.keys())
bp = ax.boxplot(cv_data, tick_labels=labels, patch_artist=True)
colors_box = ['#3498db', '#2ecc71', '#e74c3c', '#8e44ad']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('F1-Score')
ax.set_title('Comparacion de modelos - Validacion cruzada (5-fold)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, (pipe, y_pred, _)) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['Pro', 'Elite'],
        cmap='Blues', ax=ax
    )
    ax.set_title(name, fontsize=10)

fig.suptitle('Matrices de confusion - Conjunto de test', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for name, (pipe, _, _) in models.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax, name=name)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio')
ax.set_title('Curvas ROC - Comparacion de modelos')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = max(models, key=lambda k: float(models[k][2].mean()))
best_pipe, best_pred, best_cv = models[best_model_name]

print(f'\nMejor modelo: {best_model_name}')
print('='*60)
print(classification_report(y_test, best_pred, target_names=['Pro', 'Elite']))

---
## 12. Hallazgos y discusión

| Modelo | Fortalezas observadas | Limitaciones observadas |
|---|---|---|
| **Regresión Logística** | Buen F1, coeficientes interpretables: `avg_brawler_power`, `expLevel` y `total_victories` empujan hacia "Élite" | Asume relación lineal; sensible a features muy correlacionadas (ej. `avg_brawler_trophies` con el propio target) |
| **Árbol de Decisión** | Suele ser el más fuerte aquí: las reglas capturan cortes naturales (ej. umbrales de `avg_brawler_power`) | Riesgo de sobreajuste si no se limita `max_depth`; inestable ante pequeños cambios de datos |
| **SVM** | Se beneficia claramente de la estandarización (comparar F1 con/sin `StandardScaler`) | Más lento de tunear (elección de kernel); menos interpretable |
| **KNN** | Simple y razonable como baseline basado en distancias | Suele quedar por debajo de LR/DT aquí: el problema tiene una frontera relativamente "reglas" más que "vecindarios" |

**Sobre el dataset:** a diferencia de Wine Quality (features fisicoquímicas independientes), aquí varias features derivan unas de otras (ej. `avg_brawler_trophies` está mecánicamente ligada a `trophies`, que es la base del target). Esto explica los F1 altos observados y es un punto de discusión importante: un modelo "demasiado bueno" puede señalar **fuga de información (data leakage)** más que una relación genuina a explotar en producción.

---
*Ingeniería del Conocimiento (ISO56B) — UNCP — 2026-II — Ejercicio 3 (dataset propio: Brawl Stars Rankings)*